# One run in MLflow

A run is the parameters, the metrics, and the files you attach. This notebook logs one CNN fit to a folder named `tracking/`. It does not register a model. Registration is Day 4.

Then open the local UI and find the run. The experiment name is `grocery-image`.


In [ ]:
import os
import sys
from pathlib import Path

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    for folder in (candidate, candidate / "04-deep-learning"):
        marker = folder / "lib" / "customers.py"
        if marker.is_file() and (folder / "app.py").is_file():
            root = folder
            break
    if root is not None:
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from the repo folder or from 04-deep-learning.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
from pathlib import Path

import pandas as pd

from cnn import confusion, train_cnn
from cup import class_parts
from tracking import log_cnn_run

bundle, train, val, _hold_x = class_parts("auto")
learning_rate = 0.001
result = train_cnn(
    train[0],
    train[1],
    val[0],
    val[1],
    bundle["image_size"],
    len(bundle["classes"]),
    learning_rate=learning_rate,
)
matrix = confusion(result["model"], val[0], val[1], len(bundle["classes"]))
artifact = ROOT / "tracking" / "confusion.csv"
artifact.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(matrix).to_csv(artifact, index=False)
run_id = log_cnn_run(
    ROOT,
    {
        "learning_rate": learning_rate,
        "epochs": result["epochs"],
        "source": bundle["source"],
        "train_rows": len(train[0]),
    },
    {"validation_accuracy": result["validation_accuracy"], "train_accuracy": result["train_accuracy"]},
    artifact,
)
print("run", run_id)
print("validation accuracy", round(result["validation_accuracy"], 3))
print("seconds", round(result["seconds"], 2))


From this folder:

```sh
MLFLOW_ALLOW_FILE_STORE=true uv run python -m mlflow ui --backend-store-uri "file://${PWD}/tracking" --host 127.0.0.1 --port 5000
```

Open `http://127.0.0.1:5000`. The file store needs `MLFLOW_ALLOW_FILE_STORE=true`, which the notebook already set.

## Why this model

Exercise. One sentence for the share-out: would you trust this photo label on a damaged-goods desk, and what does a wrong label cost?


In [ ]:
debrief = ""  # Exercise. One sentence.
print(debrief or "Write one sentence, then run this cell again.")
